In [1]:
# house_price_prediction.py

import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Load the California Housing dataset
housing = fetch_california_housing()

X = housing.data
y = housing.target
feature_names = housing.feature_names

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create a model pipeline:
# 1. Standardize the feature values
# 2. Train a random forest with 200 trees
model = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )
)

# Train the model
model.fit(X_train, y_train)

# Evaluate it on data it has not seen before
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Model evaluation")
print(f"Mean absolute error: ${mae * 100_000:,.0f}")
print(f"Root mean squared error: ${rmse * 100_000:,.0f}")
print(f"R² score: {r2:.3f}")

# Show which features the random forest found most useful
forest = model.named_steps["randomforestregressor"]
importance = forest.feature_importances_

print("\nFeature importance")
for name, score in sorted(
    zip(feature_names, importance),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{name}: {score:.3f}")

# Predict for a new census area.
# Order: MedInc, HouseAge, AveRooms, AveBedrms,
#        Population, AveOccup, Latitude, Longitude
new_area = np.array([[8.3, 30, 6.2, 1.0, 250, 2.8, 37.8, -122.4]])

predicted_value = model.predict(new_area)[0]
print(f"\nPredicted median house value: ${predicted_value * 100_000:,.0f}")

Model evaluation
Mean absolute error: $32,712
Root mean squared error: $50,449
R² score: 0.806

Feature importance
MedInc: 0.527
AveOccup: 0.138
Latitude: 0.089
Longitude: 0.088
HouseAge: 0.054
AveRooms: 0.044
Population: 0.031
AveBedrms: 0.029

Predicted median house value: $441,621
